In [ ]:
# =====================================================================
# Student Lifestyle & Academic Productivity Survey - Preprocessing
# =====================================================================
import re, numpy as np, pandas as pd
from sklearn.impute import KNNImputer

PATH = "/content/Student Lifestyle & Academic Productivity Survey (Responses).xlsx"
raw = pd.read_excel(PATH)

# ---------- 0. SHORT COLUMN NAMES ----------
raw.columns = ["timestamp","year","gender","gym","sports","sleep_hrs","bedtime",
               "sleep_quality","phone_in_bed","screen_hrs","social_hrs","gaming_hrs",
               "distraction","study_hrs","consistency","attendance","cgpa",
               "stress","concentration","productivity","tiredness","activity_helps"]
log = []                                   # audit trail for the report
def note(msg): log.append(msg); print(msg)
note(f"Raw rows: {len(raw)}")

# ---------- 1. DROP EMPTY / UNUSABLE ROWS ----------
answers = raw.drop(columns="timestamp")
blank = answers.isna().all(axis=1)
note(f"Completely blank submissions (only timestamp): {blank.sum()} -> dropped")
df = raw[~blank].copy()

outcome_cols = ["stress","concentration","productivity"]
no_outcome = df[outcome_cols].isna().all(axis=1)
note(f"Rows with no outcome answers (cannot be imputed): {no_outcome.sum()} -> dropped")
df = df[~no_outcome].copy()

dups = df.drop(columns="timestamp").duplicated().sum()
note(f"Exact duplicate responses: {dups}")
df = df.drop_duplicates(subset=df.columns.drop("timestamp"))

# ---------- 2. CATEGORICAL STANDARDISATION ----------
df["gender"] = df["gender"].str.strip().str.title()
for c in ["year","gym","sports","bedtime","phone_in_bed","attendance","activity_helps"]:
    df[c] = df[c].str.strip()
df["year_group"] = df["year"].map({"1st year":"1st-2nd year","2nd year":"1st-2nd year",
                                   "3rd year":"3rd year","4th year":"4th year / PG",
                                   "Postgraduate":"4th year / PG"})   # merge sparse levels

# ---------- 3. FREE-TEXT NUMERIC PARSER ----------
SPECIAL = {            # hand-mapped answers that regex cannot read
    "no time":0, "o hours":0, "0h":0, "na":np.nan, "1/3":np.nan,
    "i mostly start studying only before exams":0,
    "0 hours(unless exams are approaching)":0,
    "around an hour":1, "around an hour and 30 mins":1.5, "around 1.5 hours":1.5,
    "30 mins -1hr":0.75, "around 10-20 mins":0.25, "less than 30 mins":0.25,
    "13 minutes":13/60, "1 or 2":1.5,
}
def to_hours(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if s in SPECIAL: return SPECIAL[s]
    nums = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums: return np.nan
    v = float(np.mean(nums[:2]))                 # range "5-6" / "6 to 7" -> midpoint
    if "min" in s and not re.search(r"hour|hr|\dh", s): v /= 60
    return v

num_text = ["sleep_hrs","screen_hrs","social_hrs","gaming_hrs","study_hrs"]
for c in num_text: df[c] = df[c].map(to_hours)
df["cgpa"] = pd.to_numeric(df["cgpa"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce")

# ---------- 4. IMPOSSIBLE-VALUE RULES -> NaN (do not delete the row) ----------
RULES = {"sleep_hrs":(2,14), "screen_hrs":(0,16), "social_hrs":(0,16),
         "gaming_hrs":(0,16), "study_hrs":(0,14), "cgpa":(4,10)}
for c,(lo,hi) in RULES.items():
    bad = df[c].notna() & ~df[c].between(lo,hi)
    if bad.any(): note(f"{c}: {bad.sum()} impossible value(s) {df.loc[bad,c].tolist()} -> NaN")
    df.loc[bad,c] = np.nan

# ---------- 5. LOGICAL CONSISTENCY FLAGS (flag, don't delete) ----------
df["flag_social_gt_screen"] = df["social_hrs"] > df["screen_hrs"]
df["flag_day_gt_24h"] = (df[["sleep_hrs","study_hrs","screen_hrs"]].sum(axis=1, min_count=3) > 24)
df["flag_activity_contradiction"] = (df["activity_helps"].eq("I don't do any physical activity")
                                     & ((df["gym"]!="0 days") | (df["sports"]!="0 hours")))
likert = ["sleep_quality","distraction","consistency","stress","concentration","productivity","tiredness"]
df["flag_straightline"] = df[likert].nunique(axis=1).eq(1)
for f in [c for c in df if c.startswith("flag_")]:
    note(f"{f}: {int(df[f].sum())} row(s)")

# ---------- 6. ORDINAL ENCODING OF BANDED COLUMNS ----------
gym_map    = {"0 days":0, "1-2 days":1.5, "3-4 days":3.5, "5 or more days":5.5}
sports_map = {"0 hours":0, "1-3 hours":2, "4-6 hours":5, "7 or more hours":8}
bed_map    = {"Before 11 PM":1, "11 PM - 1 AM":2, "1 AM - 3 AM":3, "After 3 AM":4}
phone_map  = {"Never":0, "Sometimes":1, "Always":2}
att_map    = {"Below 60%":1, "60-75%":2, "76-90%":3, "Above 90%":4}
df["gym_n"]=df["gym"].map(gym_map);       df["sports_n"]=df["sports"].map(sports_map)
df["bedtime_n"]=df["bedtime"].map(bed_map); df["phone_bed_n"]=df["phone_in_bed"].map(phone_map)
df["attendance_n"]=df["attendance"].map(att_map)

# ---------- 7. MISSINGNESS REPORT (after cleaning) ----------
miss = df.drop(columns=[c for c in df if c.startswith("flag_")]).isna().sum()
print("\nRemaining missing values per column:\n", miss[miss>0])
df_clean = df.copy()                       # <- analysis-ready, NO imputation

# ---------- 8. IMPUTATION (predictors only; never outcomes) ----------
imp_cols = ["sleep_hrs","screen_hrs","social_hrs","gaming_hrs","study_hrs"]
for c in imp_cols: df[f"{c}_was_missing"] = df[c].isna()          # missing-indicator flags
knn_feats = imp_cols + ["gym_n","sports_n","bedtime_n","sleep_quality","distraction","consistency"]
df_imp = df.copy()
df_imp[knn_feats] = KNNImputer(n_neighbors=5, weights="distance").fit_transform(df[knn_feats])
# CGPA and the 1-5 outcomes stay as-is: use pairwise deletion in each analysis.

# ---------- 9. FEATURE ENGINEERING ----------
def engineer(d):
    d = d.copy()
    d["activity_total"] = d["gym_n"] + d["sports_n"]
    d["activity_level"] = pd.cut(d["activity_total"], [-1,0.5,4,100], labels=["Inactive","Moderate","Active"])
    d["sleep_group"]  = pd.cut(d["sleep_hrs"],  [0,5.99,8,24], labels=["<6h","6-8h","8h+"])
    d["screen_group"] = pd.cut(d["screen_hrs"], [-1,3,6,24], labels=["Low (<=3h)","Medium (3-6h)","High (>6h)"])
    # reverse-code negative items so higher = better on every composite
    d["stress_r"]=6-d["stress"]; d["tiredness_r"]=6-d["tiredness"]; d["distraction_r"]=6-d["distraction"]
    d["academic_score"]  = d[["productivity","concentration"]].mean(axis=1)/5*100
    d["wellbeing_score"] = d[["stress_r","tiredness_r","sleep_quality"]].mean(axis=1)/5*100
    d["focus_score"]     = d[["concentration","distraction_r"]].mean(axis=1)/5*100
    return d
df_clean = engineer(df_clean); df_imp = engineer(df_imp)

# ---------- 10. RELIABILITY OF COMPOSITES (Cronbach's alpha) ----------
def cronbach(d):
    d = d.dropna(); k = d.shape[1]
    return k/(k-1)*(1 - d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
note(f"Cronbach alpha wellbeing (stress_r, tiredness_r, sleep_quality): "
     f"{cronbach(df_clean[['stress_r','tiredness_r','sleep_quality']]):.2f}")

df_clean.to_csv("cleaned_no_imputation.csv", index=False)
df_imp.to_csv("cleaned_imputed.csv", index=False)
print("\nFinal shape:", df_clean.shape)

Raw rows: 59
Completely blank submissions (only timestamp): 4 -> dropped
Rows with no outcome answers (cannot be imputed): 1 -> dropped
Exact duplicate responses: 0
sleep_hrs: 1 impossible value(s) [0.0] -> NaN
screen_hrs: 1 impossible value(s) [99.0] -> NaN
social_hrs: 1 impossible value(s) [100.0] -> NaN
gaming_hrs: 1 impossible value(s) [1000.0] -> NaN
study_hrs: 1 impossible value(s) [100.0] -> NaN
cgpa: 1 impossible value(s) [1000.0] -> NaN
flag_social_gt_screen: 3 row(s)
flag_day_gt_24h: 2 row(s)
flag_activity_contradiction: 2 row(s)
flag_straightline: 1 row(s)

Remaining missing values per column:
 sleep_hrs     6
screen_hrs    1
social_hrs    1
gaming_hrs    2
study_hrs     3
cgpa          6
dtype: int64
Cronbach alpha wellbeing (stress_r, tiredness_r, sleep_quality): 0.36

Final shape: (54, 42)


EXPLORATORY ANALYSIS AND VISULAIZATON


In [ ]:
# =====================================================================
# Part 2: EDA, Visualizations, Hypothesis Tests
# Needs: cleaned_no_imputation.csv and cleaned_imputed.csv (from preprocessing.py)
# Outputs: figures.pdf (all charts), stats_results.csv, group_tests.csv
# =====================================================================
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi":110, "axes.titleweight":"bold"})
df  = pd.read_csv("cleaned_no_imputation.csv")
dfi = pd.read_csv("cleaned_imputed.csv")
for d in (df, dfi):
    d["activity_level"] = pd.Categorical(d["activity_level"], ["Inactive","Moderate","Active"], ordered=True)
    d["sleep_group"]    = pd.Categorical(d["sleep_group"], ["<6h","6-8h","8h+"], ordered=True)
    d["screen_group"]   = pd.Categorical(d["screen_group"], ["Low (<=3h)","Medium (3-6h)","High (>6h)"], ordered=True)
pdf = PdfPages("figures.pdf")
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"{name}.png", bbox_inches="tight"); pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

# ---------- 1. SAMPLE PROFILE ----------
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c, t in zip(ax, ["year_group","activity_level","phone_in_bed"], ["Year of study","Physical activity level","Phone use before sleep"]):
    order = df[c].value_counts().index if c=="phone_in_bed" else None
    sns.countplot(data=df, x=c, ax=a, order=order if order is not None else (df[c].cat.categories if hasattr(df[c],"cat") else None))
    a.set_title(t); a.set_xlabel(""); a.bar_label(a.containers[0])
save(fig, "fig01_profile")

# ---------- 2. DISTRIBUTIONS ----------
cols = ["sleep_hrs","screen_hrs","social_hrs","study_hrs","cgpa","gaming_hrs"]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for a, c in zip(ax.ravel(), cols):
    sns.histplot(df[c].dropna(), kde=True, ax=a, bins=12); a.axvline(df[c].median(), color="red", ls="--", lw=1)
    a.set_title(f"{c} (median {df[c].median():.1f}, n={df[c].notna().sum()})")
save(fig, "fig02_distributions")

# ---------- 3. CORRELATION HEATMAP (Spearman, pairwise) ----------
key = ["gym_n","sports_n","sleep_hrs","sleep_quality","screen_hrs","social_hrs","distraction",
       "study_hrs","consistency","cgpa","stress","tiredness","concentration","productivity"]
corr = df[key].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            mask=np.triu(np.ones_like(corr, dtype=bool), 1), square=True, ax=ax, annot_kws={"size":8})
ax.set_title("Spearman correlation matrix (pairwise complete)")
save(fig, "fig03_heatmap")

# ---------- 4. PHYSICAL ACTIVITY vs OUTCOMES ----------
fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
for a, y in zip(ax, ["stress","concentration","productivity","cgpa"]):
    sns.boxplot(data=df, x="activity_level", y=y, ax=a, showfliers=False, width=.55)
    sns.stripplot(data=df, x="activity_level", y=y, ax=a, color="black", alpha=.4, size=3, jitter=.2)
    a.set_title(f"Activity level vs {y}"); a.set_xlabel("")
save(fig, "fig04_activity_outcomes")

# ---------- 5. SLEEP ----------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, y in zip(ax, ["stress","concentration","productivity"]):
    d = df.dropna(subset=["sleep_hrs", y])
    sns.regplot(data=d, x="sleep_hrs", y=y, ax=a, x_jitter=.1, y_jitter=.1,
                scatter_kws={"alpha":.5}, line_kws={"color":"red"}, lowess=True)
    a.set_title(f"Sleep hours vs {y}")
save(fig, "fig05_sleep")

# ---------- 6. SCREEN TIME ----------
fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
for a, y in zip(ax, ["productivity","concentration","distraction","stress"]):
    sns.boxplot(data=df, x="screen_group", y=y, ax=a, showfliers=False, width=.55)
    sns.stripplot(data=df, x="screen_group", y=y, ax=a, color="black", alpha=.4, size=3, jitter=.2)
    a.set_title(f"Screen time vs {y}"); a.set_xlabel(""); a.tick_params(axis="x", labelsize=8)
save(fig, "fig06_screen")

# ---------- 7. STUDY HABITS vs CGPA ----------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
sns.boxplot(data=df, x="consistency", y="cgpa", ax=ax[0], showfliers=False)
sns.stripplot(data=df, x="consistency", y="cgpa", ax=ax[0], color="black", alpha=.4, size=3)
ax[0].set_title("Study consistency vs CGPA")
d = df.dropna(subset=["study_hrs","cgpa"])
sns.regplot(data=d, x="study_hrs", y="cgpa", ax=ax[1], scatter_kws={"alpha":.6}, line_kws={"color":"red"})
ax[1].set_title("Study hours/day vs CGPA")
save(fig, "fig07_study")

# ---------- 8. BEDTIME PHONE USE & PERCEIVED BENEFIT ----------
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
sns.boxplot(data=df, x="phone_in_bed", y="sleep_quality", ax=ax[0], showfliers=False, order=["Never","Sometimes","Always"])
sns.stripplot(data=df, x="phone_in_bed", y="sleep_quality", ax=ax[0], color="black", alpha=.4, size=3, order=["Never","Sometimes","Always"])
ax[0].set_title("Phone before sleep vs sleep quality")
order = df["activity_helps"].value_counts().index
sns.countplot(data=df, y="activity_helps", order=order, ax=ax[1]); ax[1].bar_label(ax[1].containers[0])
ax[1].set_title("Do students feel activity helps their studies?"); ax[1].set_ylabel("")
save(fig, "fig08_bed_perception")

# ---------- 9. SLEEP x ACTIVITY INTERACTION ----------
piv = df.pivot_table(index="sleep_group", columns="activity_level", values="academic_score", aggfunc="mean", observed=True)
cnt = df.pivot_table(index="sleep_group", columns="activity_level", values="academic_score", aggfunc="count", observed=True)
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(piv, annot=piv.round(0).astype("Int64").astype(str) + "\n(n=" + cnt.astype("Int64").astype(str) + ")",
            fmt="", cmap="YlGnBu", ax=ax)
ax.set_title("Mean academic score by sleep x activity (cell n shown)")
save(fig, "fig09_sleep_activity")

# ---------- 10. STATISTICAL TESTS ----------
def spearman_ci(x, y, n_boot=3000, seed=42):
    d = pd.concat([x, y], axis=1).dropna(); n = len(d)
    rho, p = stats.spearmanr(d.iloc[:,0], d.iloc[:,1])
    rng = np.random.default_rng(seed); bs = []
    for _ in range(n_boot):
        s = d.sample(n, replace=True, random_state=int(rng.integers(1e9)))
        bs.append(stats.spearmanr(s.iloc[:,0], s.iloc[:,1])[0])
    lo, hi = np.nanpercentile(bs, [2.5, 97.5]); return n, rho, p, lo, hi

PAIRS = [("activity_total","stress"),("activity_total","concentration"),("activity_total","productivity"),("activity_total","cgpa"),
         ("sleep_hrs","stress"),("sleep_hrs","concentration"),("sleep_hrs","productivity"),("sleep_quality","concentration"),
         ("screen_hrs","productivity"),("screen_hrs","concentration"),("screen_hrs","distraction"),("social_hrs","concentration"),
         ("study_hrs","cgpa"),("consistency","cgpa"),("consistency","productivity"),("distraction","concentration"),
         ("phone_bed_n","sleep_quality"),("bedtime_n","concentration")]
rows = [(a, b, *spearman_ci(df[a], df[b])) for a, b in PAIRS]
res = pd.DataFrame(rows, columns=["X","Y","n","rho","p_raw","ci_low","ci_high"])
res["p_adj_BH"] = multipletests(res.p_raw, method="fdr_bh")[1]
res["strength"] = pd.cut(res.rho.abs(), [0,.1,.3,.5,1], labels=["negligible","weak","moderate","strong"])
res["sig_raw_0.05"] = res.p_raw < .05; res["sig_after_BH"] = res.p_adj_BH < .05
res.round(3).to_csv("stats_results.csv", index=False)
print("\n=== Spearman results (BH = multiple-testing correction) ==="); print(res.round(3).to_string(index=False))

grp = []
for g in ["activity_level","screen_group","sleep_group"]:
    for y in ["stress","concentration","productivity","cgpa"]:
        samples = [s[y].dropna().values for _, s in df.groupby(g, observed=True) if s[y].notna().sum() >= 3]
        if len(samples) >= 2:
            H, p = stats.kruskal(*samples); n = sum(map(len, samples)); eps2 = (H - len(samples) + 1) / (n - len(samples))
            grp.append((g, y, n, round(H,2), round(p,4), round(max(eps2,0),3)))
gt = pd.DataFrame(grp, columns=["group_var","outcome","n","H","p","epsilon_sq"])
gt["p_adj_BH"] = multipletests(gt.p, method="fdr_bh")[1].round(4); gt.to_csv("group_tests.csv", index=False)
print("\n=== Kruskal-Wallis ==="); print(gt.to_string(index=False))

# ---------- 11. REGRESSION (imputed predictors, robust SE) ----------
X = dfi[["activity_total","sleep_hrs","screen_hrs","study_hrs","consistency"]].copy()
Xz = sm.add_constant((X - X.mean()) / X.std())
for outcome in ["academic_score","stress"]:
    m = sm.OLS(dfi[outcome], Xz, missing="drop").fit(cov_type="HC3")
    print(f"\n=== OLS (standardized, HC3) -> {outcome} | n={int(m.nobs)} | R2={m.rsquared:.2f} adjR2={m.rsquared_adj:.2f} ===")
    print(m.summary2().tables[1].round(3))
vif = pd.Series([variance_inflation_factor(Xz.values, i) for i in range(1, Xz.shape[1])], index=X.columns)
print("\nVIF:", vif.round(2).to_dict())
m = sm.OLS(dfi["academic_score"], Xz).fit(cov_type="HC3")
coef = m.params.drop("const"); ci = m.conf_int().drop("const")
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(coef, range(len(coef)), xerr=[coef-ci[0], ci[1]-coef], fmt="o", capsize=4)
ax.axvline(0, color="grey", ls="--"); ax.set_yticks(range(len(coef))); ax.set_yticklabels(coef.index)
ax.set_title("Standardized effect on academic score (95% CI)")
save(fig, "fig10_regression")
pdf.close(); print("\nSaved figures.pdf, stats_results.csv, group_tests.csv")


=== Spearman results (BH = multiple-testing correction) ===
             X             Y  n    rho  p_raw  ci_low  ci_high  p_adj_BH   strength  sig_raw_0.05  sig_after_BH
activity_total        stress 54 -0.117  0.401  -0.413    0.196     0.837       weak         False         False
activity_total concentration 54  0.193  0.163  -0.090    0.454     0.731       weak         False         False
activity_total  productivity 54  0.119  0.391  -0.179    0.416     0.837       weak         False         False
activity_total          cgpa 48 -0.084  0.572  -0.350    0.189     0.837 negligible         False         False
     sleep_hrs        stress 48  0.036  0.807  -0.255    0.303     0.837 negligible         False         False
     sleep_hrs concentration 48 -0.051  0.731  -0.317    0.228     0.837 negligible         False         False
     sleep_hrs  productivity 48 -0.030  0.837  -0.319    0.275     0.837 negligible         False         False
 sleep_quality concentration 54  0.406  0.0

In [ ]:
# =====================================================================
# Part 2: EDA, Visualizations, Hypothesis Tests
# Needs: cleaned_no_imputation.csv and cleaned_imputed.csv (from preprocessing.py)
# Outputs: figures.pdf (all charts), stats_results.csv, group_tests.csv
# =====================================================================
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi":110, "axes.titleweight":"bold"})
df  = pd.read_csv("cleaned_no_imputation.csv")
dfi = pd.read_csv("cleaned_imputed.csv")
for d in (df, dfi):
    d["activity_level"] = pd.Categorical(d["activity_level"], ["Inactive","Moderate","Active"], ordered=True)
    d["sleep_group"]    = pd.Categorical(d["sleep_group"], ["<6h","6-8h","8h+"], ordered=True)
    d["screen_group"]   = pd.Categorical(d["screen_group"], ["Low (<=3h)","Medium (3-6h)","High (>6h)"], ordered=True)
pdf = PdfPages("figures.pdf")
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"{name}.png", bbox_inches="tight"); pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

# ---------- 1. SAMPLE PROFILE ----------
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c, t in zip(ax, ["year_group","activity_level","phone_in_bed"], ["Year of study","Physical activity level","Phone use before sleep"]):
    order = df[c].value_counts().index if c=="phone_in_bed" else None
    sns.countplot(data=df, x=c, ax=a, order=order if order is not None else (df[c].cat.categories if hasattr(df[c],"cat") else None))
    a.set_title(t); a.set_xlabel(""); a.bar_label(a.containers[0])
save(fig, "fig01_profile")

# ---------- 2. DISTRIBUTIONS ----------
cols = ["sleep_hrs","screen_hrs","social_hrs","study_hrs","cgpa","gaming_hrs"]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for a, c in zip(ax.ravel(), cols):
    sns.histplot(df[c].dropna(), kde=True, ax=a, bins=12); a.axvline(df[c].median(), color="red", ls="--", lw=1)
    a.set_title(f"{c} (median {df[c].median():.1f}, n={df[c].notna().sum()})")
save(fig, "fig02_distributions")

# ---------- 3. CORRELATION HEATMAP (Spearman, pairwise) ----------
key = ["gym_n","sports_n","sleep_hrs","sleep_quality","screen_hrs","social_hrs","distraction",
       "study_hrs","consistency","cgpa","stress","tiredness","concentration","productivity"]
corr = df[key].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            mask=np.triu(np.ones_like(corr, dtype=bool), 1), square=True, ax=ax, annot_kws={"size":8})
ax.set_title("Spearman correlation matrix (pairwise complete)")
save(fig, "fig03_heatmap")

# ---------- 4. PHYSICAL ACTIVITY vs OUTCOMES ----------
fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
for a, y in zip(ax, ["stress","concentration","productivity","cgpa"]):
    sns.boxplot(data=df, x="activity_level", y=y, ax=a, showfliers=False, width=.55)
    sns.stripplot(data=df, x="activity_level", y=y, ax=a, color="black", alpha=.4, size=3, jitter=.2)
    a.set_title(f"Activity level vs {y}"); a.set_xlabel("")
save(fig, "fig04_activity_outcomes")

# ---------- 5. SLEEP ----------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, y in zip(ax, ["stress","concentration","productivity"]):
    d = df.dropna(subset=["sleep_hrs", y])
    sns.regplot(data=d, x="sleep_hrs", y=y, ax=a, x_jitter=.1, y_jitter=.1,
                scatter_kws={"alpha":.5}, line_kws={"color":"red"}, lowess=True)
    a.set_title(f"Sleep hours vs {y}")
save(fig, "fig05_sleep")

# ---------- 6. SCREEN TIME ----------
fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
for a, y in zip(ax, ["productivity","concentration","distraction","stress"]):
    sns.boxplot(data=df, x="screen_group", y=y, ax=a, showfliers=False, width=.55)
    sns.stripplot(data=df, x="screen_group", y=y, ax=a, color="black", alpha=.4, size=3, jitter=.2)
    a.set_title(f"Screen time vs {y}"); a.set_xlabel(""); a.tick_params(axis="x", labelsize=8)
save(fig, "fig06_screen")

# ---------- 7. STUDY HABITS vs CGPA ----------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
sns.boxplot(data=df, x="consistency", y="cgpa", ax=ax[0], showfliers=False)
sns.stripplot(data=df, x="consistency", y="cgpa", ax=ax[0], color="black", alpha=.4, size=3)
ax[0].set_title("Study consistency vs CGPA")
d = df.dropna(subset=["study_hrs","cgpa"])
sns.regplot(data=d, x="study_hrs", y="cgpa", ax=ax[1], scatter_kws={"alpha":.6}, line_kws={"color":"red"})
ax[1].set_title("Study hours/day vs CGPA")
save(fig, "fig07_study")

# ---------- 8. BEDTIME PHONE USE & PERCEIVED BENEFIT ----------
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
sns.boxplot(data=df, x="phone_in_bed", y="sleep_quality", ax=ax[0], showfliers=False, order=["Never","Sometimes","Always"])
sns.stripplot(data=df, x="phone_in_bed", y="sleep_quality", ax=ax[0], color="black", alpha=.4, size=3, order=["Never","Sometimes","Always"])
ax[0].set_title("Phone before sleep vs sleep quality")
order = df["activity_helps"].value_counts().index
sns.countplot(data=df, y="activity_helps", order=order, ax=ax[1]); ax[1].bar_label(ax[1].containers[0])
ax[1].set_title("Do students feel activity helps their studies?"); ax[1].set_ylabel("")
save(fig, "fig08_bed_perception")

# ---------- 9. SLEEP x ACTIVITY INTERACTION ----------
piv = df.pivot_table(index="sleep_group", columns="activity_level", values="academic_score", aggfunc="mean", observed=True)
cnt = df.pivot_table(index="sleep_group", columns="activity_level", values="academic_score", aggfunc="count", observed=True)
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(piv, annot=piv.round(0).astype("Int64").astype(str) + "\n(n=" + cnt.astype("Int64").astype(str) + ")",
            fmt="", cmap="YlGnBu", ax=ax)
ax.set_title("Mean academic score by sleep x activity (cell n shown)")
save(fig, "fig09_sleep_activity")

# ---------- 10. STATISTICAL TESTS ----------
def spearman_ci(x, y, n_boot=3000, seed=42):
    d = pd.concat([x, y], axis=1).dropna(); n = len(d)
    rho, p = stats.spearmanr(d.iloc[:,0], d.iloc[:,1])
    rng = np.random.default_rng(seed); bs = []
    for _ in range(n_boot):
        s = d.sample(n, replace=True, random_state=int(rng.integers(1e9)))
        bs.append(stats.spearmanr(s.iloc[:,0], s.iloc[:,1])[0])
    lo, hi = np.nanpercentile(bs, [2.5, 97.5]); return n, rho, p, lo, hi

PAIRS = [("activity_total","stress"),("activity_total","concentration"),("activity_total","productivity"),("activity_total","cgpa"),
         ("sleep_hrs","stress"),("sleep_hrs","concentration"),("sleep_hrs","productivity"),("sleep_quality","concentration"),
         ("screen_hrs","productivity"),("screen_hrs","concentration"),("screen_hrs","distraction"),("social_hrs","concentration"),
         ("study_hrs","cgpa"),("consistency","cgpa"),("consistency","productivity"),("distraction","concentration"),
         ("phone_bed_n","sleep_quality"),("bedtime_n","concentration")]
rows = [(a, b, *spearman_ci(df[a], df[b])) for a, b in PAIRS]
res = pd.DataFrame(rows, columns=["X","Y","n","rho","p_raw","ci_low","ci_high"])
res["p_adj_BH"] = multipletests(res.p_raw, method="fdr_bh")[1]
res["strength"] = pd.cut(res.rho.abs(), [0,.1,.3,.5,1], labels=["negligible","weak","moderate","strong"])
res["sig_raw_0.05"] = res.p_raw < .05; res["sig_after_BH"] = res.p_adj_BH < .05
res.round(3).to_csv("stats_results.csv", index=False)
print("\n=== Spearman results (BH = multiple-testing correction) ==="); print(res.round(3).to_string(index=False))

grp = []
for g in ["activity_level","screen_group","sleep_group"]:
    for y in ["stress","concentration","productivity","cgpa"]:
        samples = [s[y].dropna().values for _, s in df.groupby(g, observed=True) if s[y].notna().sum() >= 3]
        if len(samples) >= 2:
            H, p = stats.kruskal(*samples); n = sum(map(len, samples)); eps2 = (H - len(samples) + 1) / (n - len(samples))
            grp.append((g, y, n, round(H,2), round(p,4), round(max(eps2,0),3)))
gt = pd.DataFrame(grp, columns=["group_var","outcome","n","H","p","epsilon_sq"])
gt["p_adj_BH"] = multipletests(gt.p, method="fdr_bh")[1].round(4); gt.to_csv("group_tests.csv", index=False)
print("\n=== Kruskal-Wallis ==="); print(gt.to_string(index=False))

# ---------- 11. REGRESSION (imputed predictors, robust SE) ----------
X = dfi[["activity_total","sleep_hrs","screen_hrs","study_hrs","consistency"]].copy()
Xz = sm.add_constant((X - X.mean()) / X.std())
for outcome in ["academic_score","stress"]:
    m = sm.OLS(dfi[outcome], Xz, missing="drop").fit(cov_type="HC3")
    print(f"\n=== OLS (standardized, HC3) -> {outcome} | n={int(m.nobs)} | R2={m.rsquared:.2f} adjR2={m.rsquared_adj:.2f} ===")
    print(m.summary2().tables[1].round(3))
vif = pd.Series([variance_inflation_factor(Xz.values, i) for i in range(1, Xz.shape[1])], index=X.columns)
print("\nVIF:", vif.round(2).to_dict())
m = sm.OLS(dfi["academic_score"], Xz).fit(cov_type="HC3")
coef = m.params.drop("const"); ci = m.conf_int().drop("const")
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(coef, range(len(coef)), xerr=[coef-ci[0], ci[1]-coef], fmt="o", capsize=4)
ax.axvline(0, color="grey", ls="--"); ax.set_yticks(range(len(coef))); ax.set_yticklabels(coef.index)
ax.set_title("Standardized effect on academic score (95% CI)")
save(fig, "fig10_regression")
pdf.close(); print("\nSaved figures.pdf, stats_results.csv, group_tests.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_no_imputation.csv'